# OpenShift Software Installer

This notebook helps you install software on OpenShift Container Platform (OCP) by:
- Reading configuration from a JSON file, or
- Getting input interactively from the user

## Prerequisites
- OpenShift CLI (`oc`) installed and configured
- Valid OpenShift cluster credentials
- Python packages: `kubernetes`, `openshift`

## 1. Install Required Dependencies

In [ ]:
!pip install kubernetes openshift pyyaml

## 2. Import Required Libraries

In [ ]:
import json
import os
import subprocess
import yaml
from pathlib import Path
from typing import Dict, Optional, List
import getpass

## 3. Configuration Class

In [ ]:
class OpenShiftConfig:
    """Configuration class for OpenShift software installation"""
    
    def __init__(self):
        self.cluster_url = None
        self.token = None
        self.namespace = None
        self.software_name = None
        self.software_image = None
        self.software_version = None
        self.replicas = 1
        self.port = 8080
        self.environment_vars = {}
        self.resource_limits = {
            'cpu': '500m',
            'memory': '512Mi'
        }
        self.resource_requests = {
            'cpu': '250m',
            'memory': '256Mi'
        }
    
    def load_from_json(self, json_file: str):
        """Load configuration from JSON file"""
        try:
            with open(json_file, 'r') as f:
                config = json.load(f)
            
            self.cluster_url = config.get('cluster_url')
            self.token = config.get('token')
            self.namespace = config.get('namespace')
            self.software_name = config.get('software_name')
            self.software_image = config.get('software_image')
            self.software_version = config.get('software_version', 'latest')
            self.replicas = config.get('replicas', 1)
            self.port = config.get('port', 8080)
            self.environment_vars = config.get('environment_vars', {})
            self.resource_limits = config.get('resource_limits', self.resource_limits)
            self.resource_requests = config.get('resource_requests', self.resource_requests)
            
            print(f"✓ Configuration loaded from {json_file}")
            return True
        except FileNotFoundError:
            print(f"✗ File not found: {json_file}")
            return False
        except json.JSONDecodeError as e:
            print(f"✗ Invalid JSON format: {e}")
            return False
    
    def get_from_user(self):
        """Get configuration interactively from user"""
        print("\n=== OpenShift Configuration ===")
        
        self.cluster_url = input("Enter OpenShift cluster URL (e.g., https://api.cluster.example.com:6443): ").strip()
        self.token = getpass.getpass("Enter OpenShift token (hidden): ").strip()
        self.namespace = input("Enter namespace/project name: ").strip()
        
        print("\n=== Software Configuration ===")
        self.software_name = input("Enter software/application name: ").strip()
        self.software_image = input("Enter container image (e.g., nginx, registry.access.redhat.com/ubi8/ubi): ").strip()
        self.software_version = input("Enter image version/tag (default: latest): ").strip() or 'latest'
        
        replicas_input = input("Enter number of replicas (default: 1): ").strip()
        self.replicas = int(replicas_input) if replicas_input else 1
        
        port_input = input("Enter container port (default: 8080): ").strip()
        self.port = int(port_input) if port_input else 8080
        
        # Optional environment variables
        add_env = input("\nAdd environment variables? (y/n, default: n): ").strip().lower()
        if add_env == 'y':
            while True:
                key = input("  Environment variable name (or press Enter to finish): ").strip()
                if not key:
                    break
                value = input(f"  Value for {key}: ").strip()
                self.environment_vars[key] = value
        
        print("\n✓ Configuration collected from user input")
    
    def validate(self) -> bool:
        """Validate required configuration"""
        required_fields = [
            ('cluster_url', self.cluster_url),
            ('token', self.token),
            ('namespace', self.namespace),
            ('software_name', self.software_name),
            ('software_image', self.software_image)
        ]
        
        missing = [field for field, value in required_fields if not value]
        
        if missing:
            print(f"✗ Missing required fields: {', '.join(missing)}")
            return False
        
        print("✓ Configuration validated successfully")
        return True
    
    def display(self):
        """Display current configuration"""
        print("\n=== Current Configuration ===")
        print(f"Cluster URL: {self.cluster_url}")
        print(f"Namespace: {self.namespace}")
        print(f"Software Name: {self.software_name}")
        print(f"Image: {self.software_image}:{self.software_version}")
        print(f"Replicas: {self.replicas}")
        print(f"Port: {self.port}")
        if self.environment_vars:
            print(f"Environment Variables: {self.environment_vars}")
        print(f"Resource Limits: {self.resource_limits}")
        print(f"Resource Requests: {self.resource_requests}")
        print("="*40)

## 4. OpenShift Installer Class

In [ ]:
class OpenShiftInstaller:
    """Handles software installation on OpenShift"""
    
    def __init__(self, config: OpenShiftConfig):
        self.config = config
    
    def login(self) -> bool:
        """Login to OpenShift cluster"""
        try:
            print("\n🔐 Logging into OpenShift cluster...")
            cmd = [
                'oc', 'login',
                self.config.cluster_url,
                '--token', self.config.token,
                '--insecure-skip-tls-verify=true'
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print("✓ Successfully logged into OpenShift cluster")
                return True
            else:
                print(f"✗ Login failed: {result.stderr}")
                return False
        except Exception as e:
            print(f"✗ Error during login: {e}")
            return False
    
    def create_namespace(self) -> bool:
        """Create or switch to namespace"""
        try:
            print(f"\n📦 Creating/switching to namespace '{self.config.namespace}'...")
            
            # Try to create namespace
            cmd = ['oc', 'new-project', self.config.namespace]
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(f"✓ Created new namespace: {self.config.namespace}")
                return True
            elif 'already exists' in result.stderr:
                # Namespace exists, switch to it
                cmd = ['oc', 'project', self.config.namespace]
                result = subprocess.run(cmd, capture_output=True, text=True)
                if result.returncode == 0:
                    print(f"✓ Switched to existing namespace: {self.config.namespace}")
                    return True
            
            print(f"✗ Failed to create/switch namespace: {result.stderr}")
            return False
        except Exception as e:
            print(f"✗ Error managing namespace: {e}")
            return False
    
    def create_deployment(self) -> bool:
        """Create deployment for the software"""
        try:
            print(f"\n🚀 Creating deployment for '{self.config.software_name}'...")
            
            # Build deployment YAML
            deployment = {
                'apiVersion': 'apps/v1',
                'kind': 'Deployment',
                'metadata': {
                    'name': self.config.software_name,
                    'labels': {
                        'app': self.config.software_name
                    }
                },
                'spec': {
                    'replicas': self.config.replicas,
                    'selector': {
                        'matchLabels': {
                            'app': self.config.software_name
                        }
                    },
                    'template': {
                        'metadata': {
                            'labels': {
                                'app': self.config.software_name
                            }
                        },
                        'spec': {
                            'containers': [{
                                'name': self.config.software_name,
                                'image': f"{self.config.software_image}:{self.config.software_version}",
                                'ports': [{
                                    'containerPort': self.config.port
                                }],
                                'resources': {
                                    'limits': self.config.resource_limits,
                                    'requests': self.config.resource_requests
                                }
                            }]
                        }
                    }
                }
            }
            
            # Add environment variables if any
            if self.config.environment_vars:
                env_list = [{'name': k, 'value': v} for k, v in self.config.environment_vars.items()]
                deployment['spec']['template']['spec']['containers'][0]['env'] = env_list
            
            # Write to temporary file
            deployment_file = f"{self.config.software_name}-deployment.yaml"
            with open(deployment_file, 'w') as f:
                yaml.dump(deployment, f)
            
            # Apply deployment
            cmd = ['oc', 'apply', '-f', deployment_file]
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(f"✓ Deployment created successfully")
                print(f"  YAML saved to: {deployment_file}")
                return True
            else:
                print(f"✗ Failed to create deployment: {result.stderr}")
                return False
        except Exception as e:
            print(f"✗ Error creating deployment: {e}")
            return False
    
    def create_service(self) -> bool:
        """Create service to expose the deployment"""
        try:
            print(f"\n🌐 Creating service for '{self.config.software_name}'...")
            
            cmd = [
                'oc', 'expose', 'deployment', self.config.software_name,
                '--port', str(self.config.port),
                '--target-port', str(self.config.port)
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0 or 'already exists' in result.stderr:
                print(f"✓ Service created/exists")
                return True
            else:
                print(f"✗ Failed to create service: {result.stderr}")
                return False
        except Exception as e:
            print(f"✗ Error creating service: {e}")
            return False
    
    def create_route(self) -> bool:
        """Create route to expose the service externally"""
        try:
            print(f"\n🔗 Creating route for '{self.config.software_name}'...")
            
            cmd = ['oc', 'expose', 'service', self.config.software_name]
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0 or 'already exists' in result.stderr:
                print(f"✓ Route created/exists")
                
                # Get route URL
                cmd = ['oc', 'get', 'route', self.config.software_name, '-o', 'jsonpath={.spec.host}']
                result = subprocess.run(cmd, capture_output=True, text=True)
                if result.returncode == 0 and result.stdout:
                    print(f"  🌍 Application URL: http://{result.stdout}")
                return True
            else:
                print(f"✗ Failed to create route: {result.stderr}")
                return False
        except Exception as e:
            print(f"✗ Error creating route: {e}")
            return False
    
    def check_status(self):
        """Check deployment status"""
        try:
            print(f"\n📊 Checking deployment status...")
            
            cmd = ['oc', 'get', 'pods', '-l', f'app={self.config.software_name}']
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(result.stdout)
            
            # Get deployment status
            cmd = ['oc', 'rollout', 'status', f'deployment/{self.config.software_name}']
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(result.stdout)
        except Exception as e:
            print(f"✗ Error checking status: {e}")
    
    def install(self) -> bool:
        """Execute full installation process"""
        print("\n" + "="*50)
        print("  OpenShift Software Installation")
        print("="*50)
        
        steps = [
            ("Login", self.login),
            ("Create/Switch Namespace", self.create_namespace),
            ("Create Deployment", self.create_deployment),
            ("Create Service", self.create_service),
            ("Create Route", self.create_route)
        ]
        
        for step_name, step_func in steps:
            if not step_func():
                print(f"\n❌ Installation failed at step: {step_name}")
                return False
        
        self.check_status()
        
        print("\n" + "="*50)
        print("  ✅ Installation completed successfully!")
        print("="*50)
        return True

## 5. Main Execution

Choose one of the following options:
- **Option A**: Load configuration from JSON file
- **Option B**: Get configuration from user input

### Option A: Load Configuration from JSON File

In [ ]:
# Initialize configuration
config = OpenShiftConfig()

# Load from JSON file
json_file = 'openshift_config.json'  # Change this to your JSON file path
if config.load_from_json(json_file):
    config.display()
else:
    print("Failed to load configuration from JSON file")

### Option B: Get Configuration from User Input

In [ ]:
# Initialize configuration
config = OpenShiftConfig()

# Get configuration from user
config.get_from_user()
config.display()

## 6. Validate Configuration

In [ ]:
# Validate configuration before proceeding
if not config.validate():
    print("\n⚠️  Please fix the configuration issues before proceeding")
else:
    print("\n✓ Configuration is valid. Ready to install!")

## 7. Execute Installation

In [ ]:
# Create installer and execute installation
installer = OpenShiftInstaller(config)
success = installer.install()

if success:
    print("\n🎉 Your software has been successfully installed on OpenShift!")
else:
    print("\n❌ Installation encountered errors. Please check the logs above.")

## 8. Additional Management Commands

Use these cells for managing your deployed application:

In [ ]:
# Scale the deployment
!oc scale deployment/{config.software_name} --replicas=3

In [ ]:
# View logs
!oc logs -l app={config.software_name} --tail=50

In [ ]:
# Delete the application
!oc delete all -l app={config.software_name}

## Example JSON Configuration File

Create a file named `openshift_config.json` with the following structure:

```json
{
  "cluster_url": "https://api.cluster.example.com:6443",
  "token": "sha256~your-token-here",
  "namespace": "my-project",
  "software_name": "my-app",
  "software_image": "nginx",
  "software_version": "latest",
  "replicas": 2,
  "port": 8080,
  "environment_vars": {
    "ENV": "production",
    "LOG_LEVEL": "info"
  },
  "resource_limits": {
    "cpu": "1000m",
    "memory": "1Gi"
  },
  "resource_requests": {
    "cpu": "500m",
    "memory": "512Mi"
  }
}
```